# Train and compare BloodMNIST baselines

Notebook này được chuyển từ `train_baselines.py` và đã được tách thành các cell nhỏ theo từng phần chức năng.

**Cách dùng nhanh**
1. Chạy cell cài đặt package nếu môi trường còn thiếu thư viện.
2. Chỉnh cell **Notebook configuration**.
3. Chạy các cell định nghĩa hàm/lớp theo thứ tự từ trên xuống.
4. Chạy cell cuối **Run experiment** để train và tạo artifacts.

> Gợi ý: bật `args.smoke_test = True` trong cell cấu hình để kiểm tra nhanh pipeline mà không cần tải MedMNIST.


## 1. Optional setup

In [ ]:
# Chạy cell này nếu môi trường Jupyter của bạn thiếu package.
# Bỏ dấu # ở dòng dưới nếu cần cài đặt.
# !pip install medmnist scikit-learn pandas matplotlib torch torchvision


## 2. Imports, constants, and baseline config

In [ ]:
from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import platform
import random
import time
import urllib.error
import urllib.request
import warnings
from contextlib import nullcontext
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Iterable

import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.exceptions import UndefinedMetricWarning
from sklearn.metrics import f1_score, roc_auc_score
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
SUMMARY_COLUMNS = [
    "model_name",
    "architecture",
    "pretrained",
    "fine_tuning_policy",
    "input_size",
    "num_classes",
    "total_params",
    "model_size_mb",
    "epoch1_train_loss",
    "epoch1_val_loss",
    "epoch1_train_acc",
    "epoch1_val_acc",
    "best_val_acc",
    "best_val_loss",
    "final_test_acc",
    "final_test_loss",
    "macro_f1",
    "macro_auc",
    "best_epoch",
    "training_time_sec",
    "seed",
    "notes",
]


@dataclass
class BaselineConfig:
    name: str
    file_stem: str
    architecture: str
    pretrained: bool
    policy: str
    input_size: int
    epochs: int
    patience: int
    clip_grad: float | None
    use_pretrained_preprocess: bool


## 3. Notebook configuration

In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    dataset="bloodmnist",
    data_root="medmnist_data",
    artifacts_dir="artifacts",
    seed=42,
    batch_size=32,
    num_workers=-1,
    device="auto",  # "auto", "cpu", or "cuda"
    medmnist_size=28,
    epochs_a=12,
    epochs_b=20,
    epochs_c=30,
    target_acc=0.80,
    limit_train=0,
    limit_val=0,
    limit_test=0,
    smoke_test=False,
    smoke_samples=96,
    no_pretrained_weights=False,
    rerun_seeds="",  # ví dụ: "42,123,3185"
)

# Kiểm tra nhanh pipeline, không tải MedMNIST:
# args.smoke_test = True
# args.no_pretrained_weights = True
# args.epochs_a = 1
# args.epochs_b = 1
# args.epochs_c = 1
# args.smoke_samples = 96

args


## 4. Dataset wrappers and custom MiniCNN

In [ ]:
class LabelSqueezeDataset(Dataset):
    def __init__(self, dataset: Dataset):
        self.dataset = dataset

    def __len__(self) -> int:
        return len(self.dataset)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        image, target = self.dataset[index]
        target_tensor = torch.as_tensor(target, dtype=torch.long).view(-1)[0]
        return image, target_tensor


class ConvertRGB:
    def __call__(self, image: Any) -> Any:
        return image.convert("RGB")


class MiniCNN(nn.Module):
    def __init__(self, in_channels: int, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.20),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


## 5. General utilities

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def choose_device(requested: str) -> torch.device:
    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested but is not available.")
        return torch.device("cuda")
    if requested == "cpu":
        return torch.device("cpu")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def resolve_num_workers(requested: int) -> int:
    if requested >= 0:
        return requested
    return 0 if platform.system().lower().startswith("win") else 2


def package_versions() -> dict[str, str]:
    versions = {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": getattr(__import__("torchvision"), "__version__", "khong xac dinh"),
    }
    for name in ["medmnist", "numpy", "pandas", "sklearn", "matplotlib"]:
        try:
            module = __import__(name)
            versions[name] = getattr(module, "__version__", "khong xac dinh")
        except Exception:
            versions[name] = "khong cai dat"
    return versions


def ensure_artifact_dirs(root: Path) -> None:
    for relative in ["plots", "checkpoints"]:
        (root / relative).mkdir(parents=True, exist_ok=True)
    root.mkdir(parents=True, exist_ok=True)


## 6. Image transforms

In [ ]:
def pretrained_transforms(train: bool) -> transforms.Compose:
    if train:
        return transforms.Compose(
            [
                ConvertRGB(),
                transforms.RandomResizedCrop(224),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.ToTensor(),
                transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
            ]
        )
    return transforms.Compose(
        [
            ConvertRGB(),
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ]
    )


def native_transforms(train: bool, image_size: int = 28) -> transforms.Compose:
    return transforms.Compose(
        [
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
        ]
    )


## 7. Dataset helpers and MedMNIST loading

In [ ]:
def cap_dataset(dataset: Dataset, limit: int, seed: int) -> Dataset:
    if limit <= 0 or limit >= len(dataset):
        return dataset
    rng = np.random.default_rng(seed)
    indices = rng.choice(len(dataset), size=limit, replace=False).tolist()
    return Subset(dataset, indices)


def medmnist_npz_name(data_flag: str, medmnist_size: int) -> str:
    suffix = "" if medmnist_size == 28 else f"_{medmnist_size}"
    return f"{data_flag}{suffix}.npz"


def file_md5(path: Path) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def ensure_medmnist_npz(info: dict[str, Any], data_flag: str, data_root: Path, medmnist_size: int) -> None:
    url_key = "url" if medmnist_size == 28 else f"url_{medmnist_size}"
    md5_key = "MD5" if medmnist_size == 28 else f"MD5_{medmnist_size}"
    url = info.get(url_key)
    expected_md5 = info.get(md5_key)
    target_path = data_root / medmnist_npz_name(data_flag, medmnist_size)

    if target_path.exists():
        if not expected_md5 or file_md5(target_path) == expected_md5:
            return
        target_path.unlink()

    if not url:
        return

    tmp_path = target_path.with_suffix(".npz.tmp")
    last_error: Exception | None = None
    for attempt in range(1, 6):
        try:
            print(f"Downloading {target_path.name} from MedMNIST source, attempt {attempt}/5")
            request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(request, timeout=300) as response, tmp_path.open("wb") as handle:
                while True:
                    chunk = response.read(1024 * 1024)
                    if not chunk:
                        break
                    handle.write(chunk)
            if expected_md5:
                actual_md5 = file_md5(tmp_path)
                if actual_md5 != expected_md5:
                    raise RuntimeError(f"MD5 mismatch: expected {expected_md5}, got {actual_md5}")
            tmp_path.replace(target_path)
            return
        except (OSError, RuntimeError, urllib.error.URLError, urllib.error.HTTPError) as exc:
            last_error = exc
            if tmp_path.exists():
                tmp_path.unlink()
            time.sleep(min(30, attempt * 5))

    raise RuntimeError(
        f"Khong tai duoc {target_path.name} sau 5 lan thu.\n"
        f"Tai thu cong tu URL nay: {url}\n"
        f"Dat file vao: {target_path}\n"
        f"MD5 mong doi: {expected_md5 or 'khong xac dinh'}\n"
        f"Loi cuoi: {last_error}"
    )


def load_medmnist_datasets(
    data_flag: str,
    data_root: Path,
    medmnist_size: int,
    use_pretrained_preprocess: bool,
) -> tuple[Dataset, Dataset, Dataset, int, int, dict[str, Any]]:
    try:
        import medmnist
        from medmnist import INFO
    except ImportError as exc:
        raise RuntimeError(
            "Package 'medmnist' is required for the real BloodMNIST run. "
            "Install with: python -m pip install medmnist"
        ) from exc

    if data_flag not in INFO:
        raise ValueError(f"Unknown MedMNIST dataset flag: {data_flag}")

    info = INFO[data_flag]
    num_classes = len(info.get("label", {}))
    if num_classes <= 0:
        raise RuntimeError('NUM_CLASSES "khong xac dinh"; missing MedMNIST label metadata.')
    in_channels = int(info.get("n_channels", 3))
    dataset_class = getattr(medmnist, info["python_class"])
    data_root.mkdir(parents=True, exist_ok=True)
    ensure_medmnist_npz(info, data_flag, data_root, medmnist_size)

    if use_pretrained_preprocess:
        train_transform = pretrained_transforms(train=True)
        eval_transform = pretrained_transforms(train=False)
    else:
        train_transform = native_transforms(train=True, image_size=28)
        eval_transform = native_transforms(train=False, image_size=28)

    train_data = dataset_class(split="train", transform=train_transform, download=True, root=str(data_root), size=medmnist_size)
    val_data = dataset_class(split="val", transform=eval_transform, download=True, root=str(data_root), size=medmnist_size)
    test_data = dataset_class(split="test", transform=eval_transform, download=True, root=str(data_root), size=medmnist_size)
    return (
        LabelSqueezeDataset(train_data),
        LabelSqueezeDataset(val_data),
        LabelSqueezeDataset(test_data),
        num_classes,
        in_channels,
        info,
    )


def load_smoke_datasets(
    samples: int,
    num_classes: int,
    use_pretrained_preprocess: bool,
) -> tuple[Dataset, Dataset, Dataset, int, int, dict[str, Any]]:
    size = (3, 28, 28)
    transform = pretrained_transforms(train=True) if use_pretrained_preprocess else native_transforms(train=True)
    eval_transform = pretrained_transforms(train=False) if use_pretrained_preprocess else native_transforms(train=False)
    train_data = datasets.FakeData(size=samples, image_size=size, num_classes=num_classes, transform=transform)
    val_data = datasets.FakeData(size=max(24, samples // 4), image_size=size, num_classes=num_classes, transform=eval_transform)
    test_data = datasets.FakeData(size=max(24, samples // 4), image_size=size, num_classes=num_classes, transform=eval_transform)
    info = {"label": {str(i): f"class_{i}" for i in range(num_classes)}, "n_channels": 3, "python_class": "FakeData"}
    return train_data, val_data, test_data, num_classes, 3, info


## 8. DataLoaders

In [ ]:
def make_loaders(
    args: argparse.Namespace,
    use_pretrained_preprocess: bool,
    seed: int,
) -> tuple[DataLoader, DataLoader, DataLoader, int, int, dict[str, Any]]:
    if args.smoke_test:
        train_data, val_data, test_data, num_classes, in_channels, info = load_smoke_datasets(
            samples=args.smoke_samples,
            num_classes=8,
            use_pretrained_preprocess=use_pretrained_preprocess,
        )
    else:
        train_data, val_data, test_data, num_classes, in_channels, info = load_medmnist_datasets(
            data_flag=args.dataset,
            data_root=Path(args.data_root),
            medmnist_size=args.medmnist_size,
            use_pretrained_preprocess=use_pretrained_preprocess,
        )

    train_data = cap_dataset(train_data, args.limit_train, seed)
    val_data = cap_dataset(val_data, args.limit_val, seed + 1)
    test_data = cap_dataset(test_data, args.limit_test, seed + 2)

    loader_kwargs = {
        "batch_size": args.batch_size,
        "num_workers": resolve_num_workers(args.num_workers),
        "pin_memory": torch.cuda.is_available(),
    }
    train_loader = DataLoader(train_data, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_data, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_data, shuffle=False, **loader_kwargs)
    return train_loader, val_loader, test_loader, num_classes, in_channels, info


## 9. Model builders and fine-tuning policies

In [ ]:
def build_resnet18(num_classes: int, pretrained: bool, allow_weight_download: bool) -> nn.Module:
    weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained and allow_weight_download else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def set_trainable_for_freeze(model: nn.Module, unfreeze_layer4: bool = False) -> None:
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True
    if unfreeze_layer4:
        for param in model.layer4.parameters():
            param.requires_grad = True


def set_trainable_for_configured(model: nn.Module, epoch: int) -> None:
    for param in model.parameters():
        param.requires_grad = False
    for param in model.fc.parameters():
        param.requires_grad = True
    for param in model.layer4.parameters():
        param.requires_grad = True
    if epoch >= 3:
        for param in model.parameters():
            param.requires_grad = True


def configured_param_groups(model: nn.Module, epoch: int) -> list[dict[str, Any]]:
    if epoch < 3:
        return [
            {"params": [p for p in model.layer4.parameters() if p.requires_grad], "lr": 3e-4},
            {"params": [p for p in model.fc.parameters() if p.requires_grad], "lr": 1e-3},
        ]
    early_layers: list[nn.Parameter] = []
    for module in [model.conv1, model.bn1, model.layer1, model.layer2, model.layer3]:
        early_layers.extend([p for p in module.parameters() if p.requires_grad])
    return [
        {"params": early_layers, "lr": 1e-4},
        {"params": [p for p in model.layer4.parameters() if p.requires_grad], "lr": 3e-4},
        {"params": [p for p in model.fc.parameters() if p.requires_grad], "lr": 1e-3},
    ]


## 10. Metric, AMP, and checkpoint helper functions

In [ ]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def model_size_mb(model: nn.Module) -> float:
    total_bytes = 0
    for tensor in model.state_dict().values():
        total_bytes += tensor.numel() * tensor.element_size()
    return total_bytes / (1024 * 1024)


def current_lr(optimizer: optim.Optimizer) -> float:
    lrs = [group["lr"] for group in optimizer.param_groups]
    return float(sum(lrs) / len(lrs))


def make_grad_scaler(device: torch.device) -> Any:
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    return torch.cuda.amp.GradScaler(enabled=device.type == "cuda")


def autocast_context(device: torch.device) -> Any:
    if device.type != "cuda":
        return nullcontext()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        return torch.amp.autocast(device_type="cuda", enabled=True)
    return torch.cuda.amp.autocast(enabled=True)


def load_checkpoint(path: Path, device: torch.device) -> dict[str, Any]:
    try:
        return torch.load(path, map_location=device, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=device)


def unpack_batch(batch: tuple[torch.Tensor, torch.Tensor], device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    images, targets = batch
    return images.to(device, non_blocking=True), targets.to(device, non_blocking=True).long().view(-1)


## 11. Train and evaluate one epoch

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
    scaler: Any,
    clip_grad: float | None,
) -> dict[str, float]:
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch in loader:
        images, targets = unpack_batch(batch, device)
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(device):
            logits = model(images)
            loss = criterion(logits, targets)
        scaler.scale(loss).backward()
        if clip_grad is not None:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * targets.size(0)
        correct += (logits.argmax(dim=1) == targets).sum().item()
        total += targets.size(0)
    return {
        "loss": running_loss / max(total, 1),
        "acc": correct / max(total, 1),
    }


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    num_classes: int,
) -> dict[str, float]:
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_targets: list[int] = []
    all_probs: list[np.ndarray] = []
    all_preds: list[int] = []
    for batch in loader:
        images, targets = unpack_batch(batch, device)
        logits = model(images)
        loss = criterion(logits, targets)
        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)
        running_loss += loss.item() * targets.size(0)
        correct += (preds == targets).sum().item()
        total += targets.size(0)
        all_targets.extend(targets.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())
        all_probs.extend(probs.detach().cpu().numpy())

    macro_f1 = float("nan")
    macro_auc = float("nan")
    if all_targets:
        macro_f1 = f1_score(all_targets, all_preds, average="macro", zero_division=0)
        try:
            y_score = np.vstack(all_probs)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", category=UndefinedMetricWarning)
                macro_auc = roc_auc_score(
                    all_targets,
                    y_score,
                    labels=list(range(num_classes)),
                    multi_class="ovr",
                    average="macro",
                )
        except ValueError:
            macro_auc = float("nan")
    return {
        "loss": running_loss / max(total, 1),
        "acc": correct / max(total, 1),
        "macro_f1": macro_f1,
        "macro_auc": macro_auc,
    }


## 12. Checkpoints and plots

In [ ]:
def save_checkpoint(path: Path, model: nn.Module, config: BaselineConfig, metrics: dict[str, Any], seed: int) -> None:
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "config": config.__dict__,
        "metrics": metrics,
        "seed": seed,
    }
    torch.save(checkpoint, path)


def plot_history(csv_path: Path, title: str, out_path: Path) -> None:
    df = pd.read_csv(csv_path)
    df = df[df["epoch"] >= 1]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(df["epoch"], df["train_loss"], label="train_loss")
    axes[0].plot(df["epoch"], df["val_loss"], label="val_loss")
    axes[0].set_title(f"{title} - Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(df["epoch"], df["train_acc"], label="train_acc")
    axes[1].plot(df["epoch"], df["val_acc"], label="val_acc")
    axes[1].set_title(f"{title} - Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_comparison(summary_csv: Path, plots_dir: Path) -> None:
    df = pd.read_csv(summary_csv)
    labels = df["model_name"].tolist()
    x = np.arange(len(labels))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - 0.18, df["best_val_acc"], width=0.36, label="best_val_acc")
    ax.bar(x + 0.18, df["final_test_acc"], width=0.36, label="final_test_acc")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.set_ylim(0, 1)
    ax.legend()
    fig.tight_layout()
    fig.savefig(plots_dir / "comparison_accuracy_bar.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(x - 0.18, df["best_val_loss"], width=0.36, label="best_val_loss")
    ax.bar(x + 0.18, df["final_test_loss"], width=0.36, label="final_test_loss")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=15, ha="right")
    ax.legend()
    fig.tight_layout()
    fig.savefig(plots_dir / "comparison_loss_bar.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


## 13. Summary table and report helpers

In [ ]:
def dataframe_to_markdown(df: pd.DataFrame) -> str:
    headers = [str(column) for column in df.columns]
    rows = []
    for _, row in df.iterrows():
        rows.append([str(row[column]) for column in df.columns])

    def clean(value: str) -> str:
        return value.replace("|", "\\|").replace("\n", " ")

    lines = [
        "| " + " | ".join(clean(header) for header in headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        lines.append("| " + " | ".join(clean(value) for value in row) + " |")
    return "\n".join(lines)


def history_to_summary(
    config: BaselineConfig,
    history: list[dict[str, Any]],
    test_metrics: dict[str, float],
    model: nn.Module,
    seed: int,
    notes: str,
) -> dict[str, Any]:
    train_rows = [row for row in history if row["epoch"] >= 1]
    if not train_rows:
        raise RuntimeError(f"No training history for {config.name}")
    epoch1 = train_rows[0]
    best = max(train_rows, key=lambda row: (row["val_acc"], -row["val_loss"]))
    total_time = sum(float(row["epoch_time_sec"]) for row in train_rows)
    return {
        "model_name": config.name,
        "architecture": config.architecture,
        "pretrained": config.pretrained,
        "fine_tuning_policy": config.policy,
        "input_size": config.input_size,
        "num_classes": int(epoch1["num_classes"]),
        "total_params": count_params(model),
        "model_size_mb": round(model_size_mb(model), 3),
        "epoch1_train_loss": epoch1["train_loss"],
        "epoch1_val_loss": epoch1["val_loss"],
        "epoch1_train_acc": epoch1["train_acc"],
        "epoch1_val_acc": epoch1["val_acc"],
        "best_val_acc": best["val_acc"],
        "best_val_loss": best["val_loss"],
        "final_test_acc": test_metrics["acc"],
        "final_test_loss": test_metrics["loss"],
        "macro_f1": test_metrics["macro_f1"],
        "macro_auc": test_metrics["macro_auc"],
        "best_epoch": int(best["epoch"]),
        "training_time_sec": round(total_time, 2),
        "seed": seed,
        "notes": notes,
    }

def write_report(artifacts_dir: Path, summary_df: pd.DataFrame, env: dict[str, Any], dataset_info: dict[str, Any]) -> None:
    table = dataframe_to_markdown(summary_df)
    best_row = summary_df.sort_values(["final_test_acc", "best_val_acc"], ascending=False).iloc[0]
    report = f"""# Bao cao baseline BloodMNIST

## Tom tat dieu hanh

Pipeline da huan luyen va so sanh 3 baseline theo yeu cau: pretrained chua cau hinh, pretrained da cau hinh, va mo hinh tu tao. Model phu hop nhat theo `final_test_acc` trong lan chay nay la **{best_row['model_name']}** voi kien truc **{best_row['architecture']}**.

## Du lieu va preprocess

- Dataset: `{dataset_info.get('python_class', 'khong xac dinh')}` tu MedMNIST/FakeData tuy theo che do chay.
- So lop: `{summary_df['num_classes'].iloc[0] if len(summary_df) else 'khong xac dinh'}`.
- Pretrained preprocess: RandomResizedCrop(224), HorizontalFlip, ImageNet normalization cho train; Resize(256), CenterCrop(224), ImageNet normalization cho val/test.
- Native preprocess: Resize 28x28 va ToTensor cho mo hinh tu tao.
- Device: `{env.get('device', 'khong xac dinh')}`.
- Seed: `{summary_df['seed'].iloc[0] if len(summary_df) else 'khong xac dinh'}`.

## So sanh 3 mo hinh

{table}

## Bieu do

![Pretrained head-only](plots/baseline_pretrained_freeze_loss_acc.png)

![Pretrained configured](plots/baseline_pretrained_configured_loss_acc.png)

![Custom MiniCNN](plots/baseline_custom_cnn_loss_acc.png)

![Accuracy comparison](plots/comparison_accuracy_bar.png)

![Loss comparison](plots/comparison_loss_bar.png)

## Pipeline

```mermaid
flowchart LR
    A[BloodMNIST tu MedMNIST] --> B[Train/Val/Test split chuan]
    B --> C[Preprocess pretrained 224 hoac native 28]
    C --> D1[Baseline A pretrained head-only]
    C --> D2[Baseline B pretrained fine-tune]
    C --> D3[Baseline C from scratch]
    D1 --> E[CSV metrics + checkpoint]
    D2 --> E
    D3 --> E
    E --> F[Bang so sanh]
    E --> G[Bieu do loss va accuracy]
    F --> H[Bao cao Markdown tieng Viet]
    G --> H
```

## Ket luan

Chon model co `final_test_acc` cao nhat neu muc tieu la do chinh xac. Neu can thoi gian train ngan, so sanh them `training_time_sec` va `model_size_mb`. Cac model khong dat 80% da duoc ghi chu trong cot `notes` cua `comparison_summary.csv`.

## Moi truong

```json
{json.dumps(env, ensure_ascii=False, indent=2)}
```
"""
    (artifacts_dir / "report.md").write_text(report, encoding="utf-8")


## 14. Optimizer rebuild policy

In [ ]:
def maybe_rebuild_optimizer_for_dynamic_policy(
    config: BaselineConfig,
    model: nn.Module,
    epoch: int,
    optimizer: optim.Optimizer | None,
    adapted_freeze: bool,
) -> tuple[optim.Optimizer, bool]:
    if config.file_stem == "baseline_pretrained_freeze":
        if optimizer is None:
            set_trainable_for_freeze(model, unfreeze_layer4=False)
            return optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4), adapted_freeze
        return optimizer, adapted_freeze

    if config.file_stem == "baseline_pretrained_configured":
        should_rebuild = optimizer is None or epoch == 3
        set_trainable_for_configured(model, epoch)
        if should_rebuild:
            return optim.AdamW(configured_param_groups(model, epoch), weight_decay=1e-4), adapted_freeze
        return optimizer, adapted_freeze

    if optimizer is None:
        return optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4), adapted_freeze
    return optimizer, adapted_freeze


## 15. Train a single baseline

In [ ]:
def train_baseline(
    args: argparse.Namespace,
    config: BaselineConfig,
    model: nn.Module,
    loaders: tuple[DataLoader, DataLoader, DataLoader],
    num_classes: int,
    device: torch.device,
    artifacts_dir: Path,
    seed: int,
) -> tuple[dict[str, Any], list[dict[str, Any]], nn.Module]:
    train_loader, val_loader, test_loader = loaders
    criterion = nn.CrossEntropyLoss()
    scaler = make_grad_scaler(device)
    model.to(device)
    optimizer: optim.Optimizer | None = None
    scheduler: optim.lr_scheduler.LRScheduler | None = None
    history: list[dict[str, Any]] = []
    best_val_acc = -math.inf
    best_val_loss = math.inf
    best_epoch = 0
    stale_epochs = 0
    notes: list[str] = []
    adapted_freeze = False

    if config.pretrained:
        epoch0 = evaluate(model, val_loader, criterion, device, num_classes)
        history.append(
            {
                "epoch": 0,
                "train_loss": float("nan"),
                "train_acc": float("nan"),
                "val_loss": epoch0["loss"],
                "val_acc": epoch0["acc"],
                "macro_f1": epoch0["macro_f1"],
                "macro_auc": epoch0["macro_auc"],
                "epoch_time_sec": 0.0,
                "learning_rate": 0.0,
                "num_classes": num_classes,
            }
        )

    for epoch in range(1, config.epochs + 1):
        optimizer, adapted_freeze = maybe_rebuild_optimizer_for_dynamic_policy(
            config=config,
            model=model,
            epoch=epoch,
            optimizer=optimizer,
            adapted_freeze=adapted_freeze,
        )
        if scheduler is None or (config.file_stem == "baseline_pretrained_configured" and epoch == 3):
            remaining = max(1, config.epochs - epoch + 1)
            if config.file_stem == "baseline_custom_cnn":
                scheduler = optim.lr_scheduler.MultiStepLR(
                    optimizer,
                    milestones=[max(1, config.epochs // 2), max(1, int(config.epochs * 0.75))],
                    gamma=0.1,
                )
            else:
                scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining)

        start = time.perf_counter()
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            clip_grad=config.clip_grad,
        )
        val_metrics = evaluate(model, val_loader, criterion, device, num_classes)
        elapsed = time.perf_counter() - start
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
            "macro_f1": val_metrics["macro_f1"],
            "macro_auc": val_metrics["macro_auc"],
            "epoch_time_sec": elapsed,
            "learning_rate": current_lr(optimizer),
            "num_classes": num_classes,
        }
        history.append(row)

        improved = (val_metrics["acc"] > best_val_acc) or (
            val_metrics["acc"] == best_val_acc and val_metrics["loss"] < best_val_loss
        )
        if improved:
            best_val_acc = val_metrics["acc"]
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            stale_epochs = 0
            save_checkpoint(
                artifacts_dir / "checkpoints" / f"{config.file_stem}_best.pt",
                model,
                config,
                {"best_val_acc": best_val_acc, "best_val_loss": best_val_loss, "best_epoch": best_epoch},
                seed,
            )
        else:
            stale_epochs += 1

        if (
            config.file_stem == "baseline_pretrained_freeze"
            and epoch == 3
            and val_metrics["acc"] < args.target_acc
            and not adapted_freeze
        ):
            set_trainable_for_freeze(model, unfreeze_layer4=True)
            optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-4, weight_decay=1e-4)
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, config.epochs - epoch))
            adapted_freeze = True
            notes.append("val_acc < 0.80 sau epoch 3; da unfreeze layer4 va giam lr head/layer4 xuong 5e-4")

        if config.file_stem == "baseline_pretrained_configured" and epoch == 2 and val_metrics["acc"] < args.target_acc:
            notes.append("val_acc < 0.80 sau epoch 2; giu augmentation vua phai va tiep tuc lich fine-tune den het epoch")

        scheduler.step()
        print(
            f"{config.file_stem} epoch {epoch}/{config.epochs} "
            f"train_loss={row['train_loss']:.4f} train_acc={row['train_acc']:.4f} "
            f"val_loss={row['val_loss']:.4f} val_acc={row['val_acc']:.4f}"
        )

        if stale_epochs >= config.patience:
            notes.append(f"early stopping tai epoch {epoch}, best_epoch={best_epoch}")
            break

    ckpt_path = artifacts_dir / "checkpoints" / f"{config.file_stem}_best.pt"
    if ckpt_path.exists():
        checkpoint = load_checkpoint(ckpt_path, device)
        model.load_state_dict(checkpoint["model_state_dict"])
    test_metrics = evaluate(model, test_loader, criterion, device, num_classes)

    metrics_csv = artifacts_dir / f"{config.file_stem}_metrics.csv"
    pd.DataFrame(history).to_csv(metrics_csv, index=False)
    plot_history(metrics_csv, config.name, artifacts_dir / "plots" / f"{config.file_stem}_loss_acc.png")

    if not notes:
        notes.append("hoan tat theo cau hinh mac dinh")
    summary = history_to_summary(config, history, test_metrics, model, seed, "; ".join(notes))
    return summary, history, model


## 16. Experiment orchestration

In [ ]:
def parse_seed_list(args: argparse.Namespace) -> list[int]:
    if not args.rerun_seeds.strip():
        return [args.seed]
    seeds = []
    for item in args.rerun_seeds.split(","):
        item = item.strip()
        if item:
            seeds.append(int(item))
    return seeds or [args.seed]


def run_for_seed(args: argparse.Namespace, seed: int, device: torch.device, artifacts_dir: Path) -> list[dict[str, Any]]:
    set_seed(seed)
    summaries: list[dict[str, Any]] = []
    dataset_info_for_report: dict[str, Any] = {}

    baseline_a = BaselineConfig(
        name="Pretrained chua cau hinh",
        file_stem="baseline_pretrained_freeze",
        architecture="ResNet-18",
        pretrained=not args.no_pretrained_weights,
        policy="Freeze backbone, chi train fc; co the unfreeze layer4 neu val_acc < 0.80 sau epoch 3",
        input_size=224,
        epochs=args.epochs_a,
        patience=3,
        clip_grad=None,
        use_pretrained_preprocess=True,
    )
    loaders_full = make_loaders(args, use_pretrained_preprocess=True, seed=seed)
    train_loader, val_loader, test_loader, num_classes, _, dataset_info_for_report = loaders_full
    model_a = build_resnet18(num_classes, pretrained=True, allow_weight_download=not args.no_pretrained_weights)
    summary_a, _, _ = train_baseline(
        args,
        baseline_a,
        model_a,
        (train_loader, val_loader, test_loader),
        num_classes,
        device,
        artifacts_dir,
        seed,
    )
    summaries.append(summary_a)

    baseline_b = BaselineConfig(
        name="Pretrained da cau hinh",
        file_stem="baseline_pretrained_configured",
        architecture="ResNet-18",
        pretrained=not args.no_pretrained_weights,
        policy="Epoch 1-2 mo layer4+fc; tu epoch 3 unfreeze full voi LR phan tang",
        input_size=224,
        epochs=args.epochs_b,
        patience=5,
        clip_grad=1.0,
        use_pretrained_preprocess=True,
    )
    model_b = build_resnet18(num_classes, pretrained=True, allow_weight_download=not args.no_pretrained_weights)
    summary_b, _, _ = train_baseline(
        args,
        baseline_b,
        model_b,
        (train_loader, val_loader, test_loader),
        num_classes,
        device,
        artifacts_dir,
        seed,
    )
    summaries.append(summary_b)

    baseline_c = BaselineConfig(
        name="Mo hinh tu tao",
        file_stem="baseline_custom_cnn",
        architecture="MiniCNN",
        pretrained=False,
        policy="MiniCNN 28x28 tu tao, train full network",
        input_size=28,
        epochs=args.epochs_c,
        patience=7,
        clip_grad=1.0,
        use_pretrained_preprocess=False,
    )
    loaders_native_full = make_loaders(args, use_pretrained_preprocess=False, seed=seed)
    train_loader_c, val_loader_c, test_loader_c, num_classes_c, in_channels_c, _ = loaders_native_full
    model_c = MiniCNN(in_channels=in_channels_c, num_classes=num_classes_c)
    summary_c, _, _ = train_baseline(
        args,
        baseline_c,
        model_c,
        (train_loader_c, val_loader_c, test_loader_c),
        num_classes_c,
        device,
        artifacts_dir,
        seed,
    )
    summaries.append(summary_c)

    env = package_versions()
    env["device"] = str(device)
    env["cuda_available"] = torch.cuda.is_available()
    if torch.cuda.is_available():
        env["cuda_device_name"] = torch.cuda.get_device_name(0)
    env["cwd"] = os.getcwd()
    (artifacts_dir / "environment.json").write_text(json.dumps(env, ensure_ascii=False, indent=2), encoding="utf-8")
    (artifacts_dir / "dataset_info.json").write_text(
        json.dumps(dataset_info_for_report, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    return summaries


## 17. Run experiment

In [ ]:
artifacts_dir = Path(args.artifacts_dir)
ensure_artifact_dirs(artifacts_dir)

device = choose_device(args.device)
seeds = parse_seed_list(args)
all_summaries: list[dict[str, Any]] = []

print(f"Using device: {device}")
print(f"Seeds: {seeds}")

for seed in seeds:
    seed_artifacts = artifacts_dir if len(seeds) == 1 else artifacts_dir / f"seed_{seed}"
    ensure_artifact_dirs(seed_artifacts)

    summaries = run_for_seed(args, seed, device, seed_artifacts)

    summary_df = pd.DataFrame(summaries).reindex(columns=SUMMARY_COLUMNS)
    summary_df.to_csv(seed_artifacts / "comparison_summary.csv", index=False)

    plot_comparison(seed_artifacts / "comparison_summary.csv", seed_artifacts / "plots")

    env = json.loads((seed_artifacts / "environment.json").read_text(encoding="utf-8"))
    dataset_info = json.loads((seed_artifacts / "dataset_info.json").read_text(encoding="utf-8"))
    write_report(seed_artifacts, summary_df, env, dataset_info)

    all_summaries.extend(summaries)

if len(seeds) > 1:
    all_df = pd.DataFrame(all_summaries).reindex(columns=SUMMARY_COLUMNS)
    all_df.to_csv(artifacts_dir / "comparison_summary_all_seeds.csv", index=False)

print(f"Done. Artifacts saved to: {artifacts_dir.resolve()}")
summary_df
